# P2: Image Classifier — Custom CNN vs Transfer Learning

**Module:** 2 — ML & Deep Learning
**Week:** 7

## Problem Statement
Classify CIFAR-10 images using CNNs. Compare a custom architecture with transfer learning (ResNet18).

## Metrics
- Overall accuracy
- Per-class precision, recall, F1
- Training time and convergence speed

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision import models

from shared.viz_utils import setup_style, save_fig
setup_style()

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

---
## 1. Load & Explore CIFAR-10

In [ ]:
# Basic transforms for exploration
transform_basic = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

train_data = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_basic)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_basic)

print(f"Training: {len(train_data)} images")
print(f"Test: {len(test_data)} images")
print(f"Image shape: {train_data[0][0].shape}")
print(f"Classes: {CLASSES}")

In [ ]:
# TODO: Show a grid of 25 random images with their labels
# Hint: Use torchvision.utils.make_grid
# Remember to unnormalize for display!

# TODO: Show class distribution (bar chart)

---
## 2. Data Augmentation

Augmentation creates variations of training images to reduce overfitting.

In [ ]:
# TODO: Define augmented transforms for training
# transform_train = transforms.Compose([
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomCrop(32, padding=4),
#     transforms.ColorJitter(brightness=0.2, contrast=0.2),
#     transforms.ToTensor(),
#     transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
# ])
#
# TODO: Reload training data with augmented transforms
# TODO: Create DataLoaders (batch_size=128)
# TODO: Show a few augmented examples side by side with originals

---
## 3. Custom CNN (From Scratch)

In [ ]:
# TODO: Define your CNN architecture
# Suggested:
# class CustomCNN(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.features = nn.Sequential(
#             nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
#             nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
#             nn.MaxPool2d(2), nn.Dropout2d(0.25),
#             
#             nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
#             nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
#             nn.MaxPool2d(2), nn.Dropout2d(0.25),
#             
#             nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
#             nn.MaxPool2d(2), nn.Dropout2d(0.25),
#         )
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.5),
#             nn.Linear(256, 10),
#         )
#     def forward(self, x):
#         return self.classifier(self.features(x))

In [ ]:
def train_model(model, train_loader, test_loader, epochs=20, lr=0.001):
    """Train a model and return training history."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
        
        scheduler.step()
        train_acc = correct / total
        test_acc = evaluate(model, test_loader)
        history['train_loss'].append(total_loss / len(train_loader))
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(train_loader):.4f}, "
                  f"Train: {train_acc:.2%}, Test: {test_acc:.2%}")
    
    elapsed = time.time() - start_time
    print(f"\nTraining time: {elapsed:.1f}s")
    return history, elapsed

def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    return correct / total

In [ ]:
# TODO: Instantiate CustomCNN and print parameter count
# TODO: Train with train_model()
# TODO: Save history as custom_history

---
## 4. Transfer Learning — ResNet18

Instead of training from scratch, use a model pretrained on ImageNet (1.2M images, 1000 classes) and adapt it to CIFAR-10.

In [ ]:
# TODO: Load pretrained ResNet18
# resnet = models.resnet18(weights='IMAGENET1K_V1')
#
# TODO: Freeze all pretrained layers
# for param in resnet.parameters():
#     param.requires_grad = False
#
# TODO: Replace the final FC layer for 10 classes
# resnet.fc = nn.Linear(resnet.fc.in_features, 10)
#
# TODO: Print trainable vs total parameters
# TODO: Train (note: CIFAR-10 is 32x32, ResNet expects 224x224)
# You may need to resize or adjust the first conv layer

---
## 5. Model Comparison

In [ ]:
# TODO: Plot training curves (loss and accuracy) for both models side by side
# TODO: Compare final test accuracy and training time
# TODO: Create a comparison table

---
## 6. Error Analysis

In [ ]:
# TODO: Using your best model:
# 1. Compute per-class accuracy
# 2. Plot confusion matrix
# 3. Show the most confused class pairs
# 4. Display misclassified examples for the worst class

---
## 7. Visualize Learned Features (Stretch)

In [ ]:
# STRETCH: Visualize first-layer filters of your CNN
# filters = model.features[0].weight.data.cpu()
# fig, axes = plt.subplots(4, 8, figsize=(12, 6))
# for i, ax in enumerate(axes.flat):
#     if i < filters.shape[0]:
#         img = filters[i].permute(1, 2, 0)
#         img = (img - img.min()) / (img.max() - img.min())
#         ax.imshow(img)
#     ax.axis('off')

---
## What I Learned
- 

## What I'd Do Differently
- 